# 05 · Massachusetts price per therm: GAF and LDAF

Every Massachusetts gas utility bills two per-therm adjustment factors on top of its base rates:

| Factor | Recovers | Set |
|---|---|---|
| **GAF** (Gas Adjustment Factor) | Cost of buying, transporting and storing gas, passed through with no markup | Each peak (Nov–Apr) and off-peak (May–Oct) season, with mid-season revisions |
| **LDAF** (Local Distribution Adjustment Factor) | Distribution program costs: energy efficiency (included from 2025/26), GSEP pipe replacement, environmental response, residential assistance and others | Annually, with revisions |

Source: Massachusetts DPU current-rates page and historical GAF/LDAF workbook (2008 onward), parsed by `src/tariffs.py`. Customer charges and base distribution rates come from each company's rate case and are not included.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, data, tariffs, plotting
plotting.use_style()
pd.set_option("display.float_format", "{:,.4f}".format)
from pathlib import Path
if not (config.RAW_DIR / tariffs.WORKBOOK_FILE).exists():
    tariffs.download()
table = tariffs.build_rate_table(config.RAW_DIR / tariffs.WORKBOOK_FILE, config.RAW_DIR / tariffs.PAGE_FILE)
m = tariffs.monthly_factors(table)
hh = tariffs.henry_hub_per_therm(data.load_raw()["henry_hub"])
print(f"{len(table):,} filed rates, {table['effective'].nunique()} revisions, "
      f"{table['effective'].min():%b %Y} to {table['effective'].max():%b %Y}")

## Rates in effect now (residential heating)

In [ ]:
res = m[m["category"] == "Residential heating"]
now = res[res["month"] == res["month"].max()].dropna(subset=["gaf_therm"])
now.set_index("company")[["gaf_therm", "ldaf_therm", "total_therm"]].sort_values("total_therm")

## One company, by component, against wholesale gas

In [ ]:
co = "National Grid – Boston Gas"
c = res[res["company"] == co].set_index("month").loc["2019":]
fig, ax = plt.subplots()
ax.bar(c.index, c["gaf_therm"], width=25, color=plotting.SERIES[0], label="GAF (supply)")
ax.bar(c.index, c["ldaf_therm"], width=25, bottom=c["gaf_therm"].where(c["ldaf_therm"] >= 0, 0),
       color=plotting.SERIES[1], label="LDAF (delivery adjustments)")
h = hh.reindex(c.index)
ax.plot(h.index, h, color=plotting.INK, lw=1.8, label="Henry Hub")
ax.set_ylabel("$/therm")
ax.set_title(f"{co}: residential heating GAF + LDAF vs. Henry Hub")
ax.legend(ncol=3, loc="upper left")
plt.show()

## How much above wholesale is the supply charge?
GAF minus the same month's Henry Hub average. The gap covers interstate pipeline capacity into New England, storage, LNG peaking supply, and reconciliation of past over- or under-recoveries.

In [ ]:
g = res.dropna(subset=["gaf_therm"]).merge(hh, left_on="month", right_index=True)
g["premium"] = g["gaf_therm"] - g["henry_hub_therm"]
g["season"] = np.where(g["month"].dt.month.isin([11, 12, 1, 2, 3, 4]), "Peak", "Off-peak")
recent = g[g["month"] >= g["month"].max() - pd.DateOffset(years=5)]
recent.pivot_table(index="company", columns="season", values="premium", aggfunc="mean").sort_values("Peak")

## LDAF over time
The delivery adjustment has grown as more program costs flow through it (pipe replacement, and energy efficiency folded in from the 2025/26 season).

In [ ]:
ld = res.dropna(subset=["ldaf_therm"])
ann = ld.groupby([ld["month"].dt.year, "company"])["ldaf_therm"].mean().unstack()
active = ann.columns[ann.loc[ann.index.max()].notna()]
fig, ax = plt.subplots()
for i, c in enumerate(active):
    ax.plot(ann.index, ann[c], marker="o", ms=3, color=plotting.SERIES[i % 8], label=c)
ax.set_ylabel("$/therm (annual average)")
ax.set_title("Residential heating LDAF by company")
ax.legend(fontsize=7, ncol=2)
plt.show()
ann.loc[[y for y in (2015, 2020, ann.index.max()) if y in ann.index], active].T

## Takeaways
*Fill in from the tables above:* which company has the highest all-in adjustment, how large the winter supply premium over Henry Hub is, and how much LDAF has grown since 2015.